# Market Data Analysis and Hype Index Construction for a Taiwan Large-Cap Stock Universe

**A TEJ-based one-year market panel and an 8-week GDELT pooled Hype Index pilot**

This report constructs a descriptive news-attention measure for a Taiwan large-cap stock universe. I first summarize the one-year TEJ market panel and then use an eight-week GDELT news sample to build pooled raw Hype and pooled market-cap-adjusted Hype. The central question is how matched news attention is distributed across stocks and industries, and how that distribution compares with market-cap concentration.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

# Path setup: the report can be opened from the repository root or from notebooks/.
CWD = Path.cwd()
REPO_ROOT = CWD if (CWD / "report").exists() else CWD.parent
RESULTS_DIR = REPO_ROOT / "report" / "results" / "pilot_8w_manual_v5"
FIGURES_DIR = REPO_ROOT / "report" / "figures" / "pilot_8w_manual_v5"

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

DISPLAY_COLUMN_RENAMES = {
    "sample": "Sample",
    "ticker": "Ticker",
    "official_chinese_name": "Chinese name",
    "official_english_name": "English name",
    "industry": "TEJ industry",
    "industry_code": "TEJ industry code",
    "industry_chinese_label": "Chinese industry label",
    "full_original_industry_label": "Full TEJ industry label",
    "stock_count": "Stock count",
    "total_news_count": "Total matched news count",
    "total_news_count_unique_urls": "Total matched news count",
    "news_count_unique_urls": "Weekly news count",
    "pooled_raw_hype": "Pooled raw Hype",
    "pooled_market_cap_weight": "Pooled market-cap weight",
    "pooled_market_cap_adjusted_hype": "Pooled cap-adjusted Hype",
    "pooled_attention_size_imbalance": "Attention-size difference",
    "pooled_sector_raw_hype": "Pooled sector raw Hype",
    "pooled_sector_market_cap_weight": "Pooled sector market-cap weight",
    "pooled_sector_market_cap_adjusted_hype": "Pooled sector cap-adjusted Hype",
    "pooled_sector_attention_size_imbalance": "Sector attention-size difference",
    "raw_hype": "Weekly raw Hype",
    "market_cap_adjusted_hype": "Weekly cap-adjusted Hype",
    "weekly_market_cap_weight": "Weekly market-cap weight",
    "weekly_return": "Weekly return",
    "weekly_realized_volatility": "Weekly realized volatility",
    "week_index": "Week",
    "week_start": "Week start",
    "week_end": "Week end",
    "nonzero_week_count": "Weeks with news",
    "zero_week_count": "Weeks without news",
    "mean_market_cap": "Average market cap",
    "mean_daily_market_cap_weight": "Average daily market-cap weight",
    "mean_daily_return": "Mean daily return",
    "daily_volatility": "Daily volatility",
    "annualized_volatility": "Annualized volatility",
    "skewness": "Skewness",
    "kurtosis": "Kurtosis",
    "min": "Minimum",
    "max": "Maximum",
    "quantile_5pct": "5% quantile",
    "quantile_95pct": "95% quantile",
    "cumulative_return": "Cumulative return",
    "trading_days": "Trading days",
}


def load_table(name: str) -> pd.DataFrame:
    return pd.read_csv(RESULTS_DIR / name)


def _formatters(frame: pd.DataFrame) -> dict[str, str]:
    percent_keywords = (
        "raw_hype",
        "market_cap_weight",
        "attention_size_imbalance",
        "daily_market_cap_weight",
        "zero_stock_ratio",
        "zero_week_ratio",
        "missing_file_ratio",
        "mean_daily_return",
        "daily_volatility",
        "annualized_volatility",
        "cumulative_return",
        "weekly_return",
        "weekly_realized_volatility",
        "quantile",
    )
    ratio_not_percent = (
        "market_cap_adjusted_hype",
        "cap_adjusted",
        "hype_computed",
    )
    formatters: dict[str, str] = {}
    for col in frame.columns:
        col_lower = str(col).lower()
        if pd.api.types.is_integer_dtype(frame[col]):
            formatters[col] = "{:,.0f}"
        elif pd.api.types.is_float_dtype(frame[col]):
            if any(key in col_lower for key in ratio_not_percent):
                formatters[col] = "{:,.3f}"
            elif any(key in col_lower for key in percent_keywords) or col_lower in {"min", "max"}:
                formatters[col] = "{:.2%}"
            elif "market_cap" in col_lower:
                formatters[col] = "{:,.0f}"
            else:
                formatters[col] = "{:,.4f}"
    return formatters


def display_frame(frame: pd.DataFrame, caption: str | None = None) -> pd.DataFrame:
    formatters = _formatters(frame)
    display_frame = frame.rename(columns=DISPLAY_COLUMN_RENAMES).copy()
    display_formatters = {
        DISPLAY_COLUMN_RENAMES.get(column, column): formatter
        for column, formatter in formatters.items()
    }
    style = display_frame.style.hide(axis="index").format(display_formatters, na_rep="")
    if caption:
        style = style.set_caption(caption)
    display(style)
    return frame


def display_table(
    name: str,
    n: int | None = None,
    columns: list[str] | None = None,
    caption: str | None = None,
) -> pd.DataFrame:
    frame = load_table(name)
    if columns is not None:
        frame = frame[[col for col in columns if col in frame.columns]]
    if n is not None:
        frame = frame.head(n)
    return display_frame(frame, caption=caption)


def report_data_source_summary() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "Source": "TEJ daily market panel",
                "Period": "2025-04-01 to 2026-03-31",
                "Frequency": "Trading day",
                "Main fields": "Market cap, volume, market-cap weight",
                "Use in report": "Market structure and size benchmark",
            },
            {
                "Source": "TEJ adjusted return data",
                "Period": "2025-04-01 to 2026-03-31",
                "Frequency": "Trading day",
                "Main fields": "Adjusted close, simple return, log return",
                "Use in report": "Market return statistics and realized volatility",
            },
            {
                "Source": "TEJ company and industry metadata",
                "Period": "Universe selection date 2025-03-31",
                "Frequency": "Fixed cross-section",
                "Main fields": "Ticker, company names, TEJ industry labels",
                "Use in report": "Universe description and industry aggregation",
            },
            {
                "Source": "GDELT news metadata",
                "Period": "2025-04-05 to 2025-05-30",
                "Frequency": "Calendar-day news aggregated to weeks",
                "Main fields": "Company-matched news-document counts",
                "Use in report": "Pooled raw Hype and weekly Hype dynamics",
            },
            {
                "Source": "Weekly Hype analysis table",
                "Period": "2025-04-05 to 2025-05-30",
                "Frequency": "Saturday-to-Friday weeks",
                "Main fields": "Weekly news counts, market-cap weights, raw and cap-adjusted Hype",
                "Use in report": "Pooled summaries, heatmaps, and case studies",
            },
        ]
    )


def tej_coverage_summary() -> pd.DataFrame:
    raw = load_table("tej_market_validation_summary.csv")
    lookup = dict(zip(raw["metric"], raw["value"]))
    return pd.DataFrame(
        [
            ("Coverage status", "Complete for the prepared market background"),
            ("Stocks in fixed universe", int(lookup.get("daily_ticker_count", 50))),
            ("Trading dates in market panel", int(lookup.get("trading_date_count", 243))),
            ("Daily stock-date rows", int(lookup.get("daily_market_panel_rows", 12150))),
            ("Weekly market-cap dates", int(lookup.get("weekly_count", 52))),
            ("Weekly stock-weight rows", int(lookup.get("weekly_market_cap_weight_rows", 2600))),
            ("First observed trading date", lookup.get("daily_min_date", "")),
            ("Last observed trading date", lookup.get("daily_max_date", "")),
        ],
        columns=["Item", "Value"],
    )


def news_collection_summary() -> pd.DataFrame:
    raw = load_table("news_acquisition_summary.csv")
    segment_names = {
        "chunk_1": "Collection window 1",
        "chunk_2": "Collection window 2",
        "combined": "Full news window",
    }
    frame = raw.assign(Segment=raw["chunk"].map(segment_names).fillna(raw["chunk"]))
    frame = frame.rename(
        columns={
            "start_date": "Start date",
            "end_date": "End date",
            "candidate_files": "Expected GDELT files",
            "files_processed": "Processed files",
            "files_missing": "Missing files",
            "files_failed": "Failed files",
            "matched_rows": "Matched stock-news rows",
        }
    )
    return frame[
        [
            "Segment",
            "Start date",
            "End date",
            "Expected GDELT files",
            "Processed files",
            "Missing files",
            "Failed files",
            "Matched stock-news rows",
        ]
    ]


def hype_construction_summary() -> pd.DataFrame:
    return pd.DataFrame(
        [
            ("News window", "2025-04-05 to 2025-05-30"),
            ("Weekly convention", "Saturday-to-Friday weeks"),
            ("Universe size", "50 fixed stocks"),
            ("Stock-week observations", "400"),
            ("Weeks with positive news totals", "8 of 8"),
            ("Cross-sectional Hype definition", "Pooled over the full eight-week window"),
            ("Size benchmark", "Pooled market-cap share over the same weeks"),
        ],
        columns=["Item", "Value"],
    )


def return_statistics_summary() -> pd.DataFrame:
    frame = load_table("basic_return_statistics_full_vs_hype_window.csv").copy()
    frame["sample"] = frame["sample"].replace(
        {
            "full_market_background": "Full one-year market background",
            "hype_trading_window": "Hype trading comparison window",
        }
    )
    return frame


def hype_correlation_summary() -> pd.DataFrame:
    frame = load_table("hype_return_volatility_correlation_summary.csv").copy()
    frame["variable"] = frame["variable"].replace(
        {
            "raw_hype": "Weekly raw Hype",
            "market_cap_adjusted_hype": "Weekly cap-adjusted Hype",
            "news_count_unique_urls": "Weekly news count",
        }
    )
    frame["target"] = frame["target"].replace(
        {
            "weekly_return": "Weekly return",
            "weekly_realized_volatility": "Weekly realized volatility",
        }
    )
    frame = frame[
        [
            "variable",
            "target",
            "pearson_correlation",
            "spearman_correlation",
            "n_observations",
        ]
    ].rename(
        columns={
            "variable": "Measure",
            "target": "Market outcome",
            "pearson_correlation": "Pearson correlation",
            "spearman_correlation": "Spearman correlation",
            "n_observations": "Observations",
        }
    )
    return frame


def display_figure(filename: str, caption: str | None = None) -> None:
    display(Image(filename=str(FIGURES_DIR / filename)))
    if caption:
        display(Markdown(f"*{caption}*"))


assert RESULTS_DIR.is_dir(), RESULTS_DIR
assert FIGURES_DIR.is_dir(), FIGURES_DIR


## 0. Executive Summary

The empirical design uses a **fixed TEJ-based top-50 market-cap universe** and a short GDELT news pilot. Market variables are summarized over a one-year TEJ panel, while Hype Index results are computed from the eight-week news sample. The pooled Hype definitions treat the full eight-week window as a single measurement period, which matches the report objective of describing total pilot-window attention rather than an average of weekly ratios.

The prepared results show a clear concentration of news attention in technology-related stocks and industries. TSMC receives the largest pooled raw news-attention share, while MediaTek has the highest stock-level pooled market-cap-adjusted Hype among the leading stocks. At the industry level, semiconductor firms dominate total news attention, while the computer and peripheral industry receives a large attention share relative to its pooled market-cap weight. These results describe the pilot window and provide a basis for later extensions with a longer sample.


In [ ]:
executive_summary = pd.DataFrame(
    [
        ("Universe", "TEJ-based fixed top-50 market-cap listed stocks"),
        ("Universe selection date", "2025-03-31"),
        ("Market data period", "2025-04-01 to 2026-03-31"),
        ("News window", "2025-04-05 to 2025-05-30"),
        ("Market comparison window", "2025-04-07 to 2025-05-29 trading days"),
        ("Hype frequency", "Weekly bins, Saturday-to-Friday"),
        ("Cross-sectional Hype method", "Pooled raw Hype and pooled market-cap-adjusted Hype"),
        ("Research scope", "Descriptive market-data analysis and Hype Index construction"),
    ],
    columns=["Item", "Value"],
)
display(executive_summary.style.hide(axis="index"))


## 1. Introduction and Research Scope

Financial news attention is not distributed uniformly. Larger firms tend to appear more often in news coverage, but some firms or industries may receive more attention than their market size alone would suggest. This report adapts the Hype Index idea to a Taiwan large-cap setting by comparing news-attention shares with market-cap shares.

The project is best viewed as a Taiwan-market adaptation rather than an exact replication of the original U.S. study. The universe is a TEJ-based fixed top-50 market-cap proxy, the news source is GDELT metadata, and the Hype window is an eight-week pilot. The analysis therefore emphasizes construction, validation, and interpretation of the attention measures rather than trading performance or predictive modeling.

The main output is a pair of pooled indices:

- **pooled raw Hype**, the share of total matched news attention received by a stock or industry during the pilot window;
- **pooled market-cap-adjusted Hype**, the ratio of pooled attention share to pooled market-cap share.


## 2. Data Sources and Sample Alignment

The report combines market and news data with different calendars. TEJ returns and market capitalization are observed on trading days, while GDELT news metadata is organized by publication timestamps and calendar dates. The GDELT news window runs from **2025-04-05** to **2025-05-30**, while the corresponding TEJ trading comparison window runs from **2025-04-07** to **2025-05-29**. This alignment treats 2025-05-30 as a news date while keeping the market comparison on actual TEJ trading dates.

The tables below summarize the data sources and coverage checks used in the report.


In [ ]:
_ = display_frame(report_data_source_summary(), caption="Data sources used in the report")

In [ ]:
_ = display_frame(tej_coverage_summary(), caption="TEJ market-data coverage summary")

The market-data coverage summary shows 50 stocks over 243 trading dates, with 12,150 daily stock-date rows and 52 weekly market-cap dates. This one-year panel gives the report enough market background to describe size concentration, cumulative returns, and rolling volatility around the shorter news window.


## 3. Universe and Market Structure

The universe is a fixed top-50 listed-stock universe selected from TEJ market-cap data as of 2025-03-31. Keeping the stock universe fixed makes the weekly comparisons consistent: each week is evaluated against the same set of 50 companies.

TEJ industry labels include Chinese text. The figures use industry codes to keep labels compact, while the tables keep the full Chinese labels for interpretation.


In [ ]:
_ = display_table(
    "universe_summary.csv",
    n=10,
    columns=[
        "ticker",
        "official_chinese_name",
        "official_english_name",
        "industry",
        "mean_market_cap",
        "mean_daily_market_cap_weight",
    ],
    caption="Selected rows from the fixed TEJ-based top-50 universe",
)


In [ ]:
_ = display_table(
    "tej_industry_label_mapping.csv",
    columns=[
        "industry_code",
        "industry_chinese_label",
        "full_original_industry_label",
        "stock_count",
        "pooled_sector_raw_hype",
        "pooled_sector_market_cap_weight",
        "pooled_sector_market_cap_adjusted_hype",
    ],
    caption="TEJ industry code and Chinese label mapping",
)


In [ ]:
display_figure(
    "sector_stock_count.png",
    "The fixed top-50 universe is spread across TEJ industry groups, with several industries represented by only one or two stocks.",
)
display_figure(
    "sector_market_cap_weight.png",
    "Market-cap concentration is much stronger than the stock-count distribution, reflecting the dominant role of semiconductor and related large-cap firms.",
)


The market structure matters for the Hype interpretation. A large raw news share for a dominant industry may be consistent with its size, while a smaller industry can still have high market-cap-adjusted Hype if its news share is large relative to its pooled market-cap weight.


## 4. Basic Quantitative Market Analysis

The market background uses the one-year TEJ adjusted return panel. I use the simple return and log return fields from the TEJ adjusted-price export after converting percent fields into return fractions.

For stock $i$ on trading day $t$, the simple return is

$$
R_{i,t}=\frac{P^{adj}_{i,t}}{P^{adj}_{i,t-1}}-1,
$$

and the log return is

$$
r_{i,t}=\log P^{adj}_{i,t}-\log P^{adj}_{i,t-1}.
$$

As a simple market proxy for the 50-stock universe, I use the equal-weight daily return

$$
R_t^{EW}=\frac{1}{N}\sum_{i=1}^{N}R_{i,t},\qquad N=50.
$$

For a set of trading dates $\mathcal{T}$, the return-statistics table reports the sample mean, sample volatility, and annualized volatility of this equal-weight market proxy:

$$
\bar R_{\mathcal{T}}^{EW}=\frac{1}{|\mathcal{T}|}\sum_{t\in\mathcal{T}}R_t^{EW},
$$

$$
s_{\mathcal{T}}^{EW}=\sqrt{\frac{1}{|\mathcal{T}|-1}\sum_{t\in\mathcal{T}}\left(R_t^{EW}-\bar R_{\mathcal{T}}^{EW}\right)^2},
\qquad
\sigma_{\mathcal{T}}^{ann}=\sqrt{252}\,s_{\mathcal{T}}^{EW}.
$$

The cumulative return is

$$
CR_t=\prod_{\tau\le t}(1+R_{\tau}^{EW})-1.
$$

The rolling-volatility figure applies the same annualization convention to a moving 20-trading-day window:

$$
\sigma_{t,20}^{ann}=\sqrt{252}\,\operatorname{Std}(R^{EW}_{t-19},\ldots,R^{EW}_{t}).
$$

The table therefore summarizes volatility over fixed samples, while the rolling-volatility figure shows how the annualized short-term market volatility changes through time.

In [ ]:
_ = display_frame(return_statistics_summary(), caption="Equal-weight return statistics: one-year background versus Hype trading window")

In [ ]:
display_figure(
    "equal_weight_cumulative_return_hype_window.png",
    "The shaded region marks the Hype trading comparison window inside the one-year equal-weight market proxy.",
)
display_figure(
    "equal_weight_rolling_volatility_20d_hype_window.png",
    "The rolling-volatility chart uses annualized 20-trading-day volatility of the equal-weight market proxy, providing market background for the shorter news-attention window.",
)


The Hype trading window has a much higher equal-weight annualized volatility than the full one-year background in the summary table. This provides useful context for interpreting attention measures. A more formal volatility study would require a longer Hype sample than the eight-week window used here.


## 5. News Acquisition and Matching Validation

The news sample covers the full calendar window from **2025-04-05** to **2025-05-30**. The acquisition summary reports 5,376 expected 15-minute GDELT files for this window, of which 5,372 were available and read. Four archives were unavailable, which represents a very small coverage gap, and no file-read errors were recorded.

The Hype measures use stock-day matched GDELT document counts as the article-attention proxy. This construction captures a reproducible measure of company-level media attention, while a complete article-level reconstruction across all stocks and weeks would require additional source-level processing. A single article can also contribute to more than one stock when multiple companies are matched in the same document. These conventions are part of the interpretation of the Hype results.


In [ ]:
_ = display_frame(news_collection_summary(), caption="GDELT news collection summary")

In [ ]:
_ = display_frame(hype_construction_summary(), caption="Hype Index construction summary")

In [ ]:
display_figure(
    "weekly_news_counts.png",
    "Weekly matched news-count volume over the eight Saturday-to-Friday Hype weeks.",
)


Every week has a positive news denominator, so the weekly Hype measures are defined throughout the pilot. The weekly totals vary substantially, which motivates using pooled ratios over the full window for cross-sectional comparisons.


## 6. Pooled Hype Index Methodology

Let $U$ denote the fixed 50-stock universe and $S$ denote the eight-week pilot window. For stock $i$ in week $w$, let $N_{i,w}$ be the weekly news count and

$$
N_w=\sum_{j\in U}N_{j,w}
$$

be the weekly universe news count. Let $MC_{i,w}$ be the weekly TEJ market-cap snapshot used for the size comparison. The corresponding weekly market-cap weight is

$$
M_{i,w}=\frac{MC_{i,w}}{\sum_{j\in U}MC_{j,w}}.
$$

These weekly primitives define the weekly Hype measures used in the heatmaps and case-study panels:

$$
H_{i,w}=\frac{N_{i,w}}{N_w},
\qquad
A_{i,w}=\frac{H_{i,w}}{M_{i,w}}.
$$

The pooled raw Hype of stock $i$ over the full pilot window is

$$
H_i^{pool}=\frac{\sum_{w\in S}N_{i,w}}{\sum_{w\in S}\sum_{j\in U}N_{j,w}}.
$$

This index measures the share of all matched pilot-window news attention received by stock $i$. The pooled market-cap weight is

$$
M_i^{pool}=\frac{\sum_{w\in S}MC_{i,w}}{\sum_{w\in S}\sum_{j\in U}MC_{j,w}}.
$$

The pooled market-cap-adjusted Hype is then

$$
A_i^{pool}=\frac{H_i^{pool}}{M_i^{pool}}.
$$

Values above one indicate that a stock's pooled news-attention share is larger than its pooled market-cap share. Values below one indicate attention below the size benchmark.

The pooled definition is chosen because the report treats the eight-week pilot as one measurement period. In general,

$$
\frac{1}{T}\sum_{w=1}^{T}\frac{N_{i,w}}{N_w}
\neq
\frac{\sum_{w=1}^{T}N_{i,w}}{\sum_{w=1}^{T}N_w}.
$$

The pooled raw Hype can also be written as a news-volume-weighted average of weekly raw Hype:

$$
H_i^{pool}=\sum_{w=1}^{T}\omega_w H_{i,w},\qquad
\omega_w=\frac{N_w}{\sum_{s\in S}N_s}.
$$

This identity makes clear why the pooled result differs from an equal-week arithmetic mean when weekly total news volume changes across the pilot.

## 7. Stock-Level Pooled Hype Results

The stock-level results separate two questions. Pooled raw Hype asks which stocks receive the largest share of matched news attention. Pooled market-cap-adjusted Hype asks which stocks receive more or less attention relative to their pooled market-cap weight.


In [ ]:
_ = display_table(
    "stock_pooled_hype_summary.csv",
    n=10,
    columns=[
        "ticker",
        "official_chinese_name",
        "official_english_name",
        "industry",
        "total_news_count_unique_urls",
        "pooled_raw_hype",
        "pooled_market_cap_weight",
        "pooled_market_cap_adjusted_hype",
        "pooled_attention_size_imbalance",
    ],
    caption="Stock-level pooled Hype summary, sorted by total attention",
)


In [ ]:
_ = display_table(
    "top_pooled_raw_hype_stocks.csv",
    caption="Top 10 stocks by pooled raw Hype",
)


In [ ]:
_ = display_table(
    "top_pooled_market_cap_adjusted_hype_stocks.csv",
    caption="Top 10 stocks by pooled market-cap-adjusted Hype",
)


TSMC has the largest pooled raw Hype share in the pilot window, but its pooled attention share is below its pooled market-cap weight. MediaTek, Hon Hai, ASUS, Formosa Plastics, and Pegatron show high attention relative to their size benchmarks. These differences illustrate why the raw and market-cap-adjusted views are both useful.


In [ ]:
display_figure(
    "top_pooled_raw_hype_stocks.png",
    "Top pooled raw Hype ranks stocks by total matched attention share over the full pilot window.",
)
display_figure(
    "top_pooled_market_cap_adjusted_hype_stocks.png",
    "The cap-adjusted ranking highlights stocks whose attention share is high relative to their pooled market-cap share.",
)
display_figure(
    "pooled_raw_hype_vs_pooled_market_cap_weight_scatter.png",
    "The proportionality line represents equal news-attention and market-cap shares.",
)
display_figure(
    "pooled_raw_hype_vs_pooled_market_cap_weight_scatter_zoom.png",
    "The zoomed view improves readability for stocks outside the dominant TSMC scale.",
)
display_figure(
    "pooled_attention_size_imbalance_top_stocks.png",
    "Attention-size imbalance is computed as pooled raw Hype minus pooled market-cap weight.",
)


Points above the proportionality line receive a larger share of news attention than their pooled market-cap share. Points below the line receive less attention relative to size. The imbalance chart shows the difference between attention share and size share, while the cap-adjusted Hype ratio shows their relative magnitude.


## 8. Sector-Level Pooled Hype Results

The same pooled construction can be applied to TEJ industry groups. For industry $G$, pooled sector raw Hype is

$$
H_G^{pool}=\frac{\sum_{i\in G}\sum_{w\in S}N_{i,w}}{\sum_{w\in S}\sum_{j\in U}N_{j,w}},
$$

pooled sector market-cap weight is

$$
M_G^{pool}=\frac{\sum_{i\in G}\sum_{w\in S}MC_{i,w}}{\sum_{w\in S}\sum_{j\in U}MC_{j,w}},
$$

and pooled sector market-cap-adjusted Hype is

$$
A_G^{pool}=\frac{H_G^{pool}}{M_G^{pool}}.
$$

The pooled raw sector Hype has a natural additivity property:

$$
H_G^{pool}=\sum_{i\in G}H_i^{pool}.
$$

This property makes the sector table a direct aggregation of the stock-level attention shares.


In [ ]:
_ = display_table(
    "sector_pooled_hype_summary.csv",
    columns=[
        "industry_code",
        "industry_chinese_label",
        "stock_count",
        "total_news_count",
        "pooled_sector_raw_hype",
        "pooled_sector_market_cap_weight",
        "pooled_sector_market_cap_adjusted_hype",
        "pooled_sector_attention_size_imbalance",
    ],
    caption="Sector-level pooled Hype summary",
)


In [ ]:
_ = display_table(
    "top_pooled_sector_market_cap_adjusted_hype.csv",
    caption="Top sectors by pooled market-cap-adjusted Hype",
)


The semiconductor industry receives the largest pooled raw sector attention share, which is broadly consistent with its dominant market-cap role in this universe. The computer and peripheral industry has a much larger attention share than its pooled market-cap weight, leading to a high sector-level attention-to-size ratio. Financial holding companies show the opposite pattern in this pilot: their pooled size share is much larger than their matched news-attention share.


In [ ]:
display_figure(
    "pooled_sector_raw_hype_vs_pooled_market_cap_weight_scatter.png",
    "Each point compares an industry group's share of total pilot-window news attention with its pooled market-cap share.",
)
display_figure(
    "pooled_sector_raw_hype_vs_pooled_market_cap_weight_scatter_zoom.png",
    "The zoomed view focuses on smaller industry groups; full Chinese labels are provided in the mapping table.",
)
display_figure(
    "pooled_sector_market_cap_adjusted_hype_ranking.png",
    "The sector cap-adjusted ranking emphasizes attention relative to pooled sector size.",
)
display_figure(
    "pooled_sector_attention_size_imbalance.png",
    "The sector imbalance chart shows pooled attention share minus pooled market-cap share.",
)

The figures use TEJ industry codes so that labels remain compact. The Chinese industry names in the mapping table above provide the corresponding full labels for discussion.


## 9. Weekly Hype Dynamics

The cross-sectional rankings use pooled Hype, but weekly Hype remains useful for inspecting time variation. The heatmaps show how attention shares changed across the eight weeks for selected high-attention stocks and high cap-adjusted-Hype stocks.


In [ ]:
display_figure(
    "raw_hype_heatmap_top_stocks.png",
    "Weekly raw Hype dynamics for selected high-news stocks.",
)
display_figure(
    "market_cap_adjusted_hype_heatmap_top_stocks.png",
    "Weekly market-cap-adjusted Hype dynamics for selected high attention-to-size stocks.",
)


The heatmaps complement the pooled cross-sectional results by showing whether attention was persistent across the pilot or concentrated in a small number of weeks.


## 10. Descriptive Relation with Returns and Volatility

The original Hype Index paper studies relationships between Hype, returns, volatility, and market risk indicators such as VIX. This report gives a narrower comparison because the news window contains only eight weeks. The analysis below describes stock-week attention measures together with returns and realized volatility in the same week.

For stock $i$ and week $w$, weekly simple return is

$$
R_{i,w}=\prod_{t\in D_w}(1+R_{i,t})-1,
$$

where $D_w$ is the set of TEJ trading days in week $w$. Weekly realized volatility is computed from log returns:

$$
RV_{i,w}=\sqrt{\sum_{t\in D_w}r_{i,t}^{2}}.
$$

This weekly realized volatility is measured on the same weekly horizon as the Hype variables and is not annualized. It is separate from the annualized 20-day rolling volatility in the market-background section, which describes the equal-weight universe through time.

In [ ]:
_ = display_frame(hype_correlation_summary(), caption="Same-week association summary for Hype, returns, and realized volatility")

The correlations with weekly return are close to zero in this pilot. The rank correlations with realized volatility are positive but still modest. This pattern is consistent with interpreting Hype as an attention measure rather than a return model.


In [ ]:
display_figure(
    "hype_vs_weekly_return_scatter.png",
    "Weekly raw Hype and same-week return.",
)
display_figure(
    "hype_vs_weekly_volatility_scatter.png",
    "Weekly raw Hype and same-week realized volatility.",
)
display_figure(
    "cap_adjusted_hype_vs_weekly_return_scatter.png",
    "Weekly market-cap-adjusted Hype and same-week return.",
)
display_figure(
    "cap_adjusted_hype_vs_weekly_volatility_scatter.png",
    "Weekly market-cap-adjusted Hype and same-week realized volatility.",
)


The cap-adjusted Hype plots contain a few large attention-to-size observations, which is expected for ratio-based measures when relatively small size benchmarks meet nonzero news counts. The scatter plots are most useful for locating same-week associations and outliers in the pilot sample.


## 11. Key Stock Case Studies

The following four case studies illustrate how weekly news counts, weekly raw Hype, weekly market-cap weight, cap-adjusted Hype, return, and realized volatility appear together for selected stocks. They are descriptive examples chosen from the high-attention portion of the universe.

- **2330 台積電 / TSMC** has the largest pooled raw Hype share and also the largest pooled market-cap weight. Its case illustrates why raw attention and size-adjusted attention can lead to different interpretations.
- **2454 聯發科 / MediaTek** combines high raw attention with a much smaller pooled size benchmark than TSMC, producing a high pooled market-cap-adjusted Hype value.
- **2317 鴻海 / Hon Hai / Foxconn** is a large electronics and supply-chain firm with substantial matched global media exposure.
- **2357 華碩 / ASUS** provides a consumer electronics example with lower raw count than the top three stocks but a high attention-to-size ratio in the pilot window.


In [ ]:
_ = display_table(
    "key_stock_case_study_summary.csv",
    n=12,
    columns=[
        "ticker",
        "official_chinese_name",
        "pooled_raw_hype",
        "pooled_market_cap_weight",
        "pooled_market_cap_adjusted_hype",
        "week_index",
        "news_count_unique_urls",
        "raw_hype",
        "market_cap_adjusted_hype",
        "weekly_return",
        "weekly_realized_volatility",
    ],
    caption="Selected rows from the weekly key-stock case-study table",
)


In [ ]:
display_figure("key_stock_case_study_2330.png", "TSMC weekly case-study panels.")
display_figure("key_stock_case_study_2454.png", "MediaTek weekly case-study panels.")
display_figure("key_stock_case_study_2317.png", "Hon Hai weekly case-study panels.")
display_figure("key_stock_case_study_2357.png", "ASUS weekly case-study panels.")


These panels are useful for reading the pooled results together with weekly paths. For example, a high pooled attention-to-size ratio can come from persistent weekly attention or from a smaller number of weeks with very high cap-adjusted Hype.


## 12. Limitations

Several design constraints shape the interpretation of the results. The news sample covers only eight weeks, which is shorter than the full sample used in the original Hype Index study. GDELT provides a reproducible global-news metadata source, while local Taiwan financial-news coverage may differ from this global source. The alias-matching procedure was manually reviewed for precision, yet company-name matching can still create false positives and false negatives.

The news count is also an operational proxy based on stock-day matched GDELT documents rather than a complete article-level reconstruction across the full universe. The return and volatility comparisons are same-week descriptions. A longer Hype sample and a separate empirical setup would be needed for lagged volatility or forecasting analysis.


## 13. Conclusion

This report builds a reproducible pooled Hype Index pilot for a Taiwan large-cap stock universe. The pooled raw Hype results describe how matched news attention is concentrated across stocks and industries during the eight-week window. The pooled market-cap-adjusted Hype results compare that attention with a size benchmark, highlighting stocks and industries whose news share is high relative to their market-cap share.

The pilot also shows why a market-data background is useful. The Hype window occurs inside a one-year TEJ market panel with visibly different volatility conditions from the full sample. Future work could extend the sample length, compare GDELT with Taiwan-specific financial news sources, and then examine lagged relationships between attention and realized volatility with a more formal empirical design.


## 14. References

1. Cao, Z., Wunkaew, W., and Geman, H. (2025). *The Hype Index: an NLP-driven Measure of Market News Attention*. arXiv:2506.06329. https://arxiv.org/abs/2506.06329
2. GDELT Project. *GDELT 2.0 and Global Knowledge Graph data documentation*. https://www.gdeltproject.org/data.html
3. Taiwan Economic Journal (TEJ) local exports used in this project for market capitalization, adjusted returns, and company metadata.
4. Project repository: https://github.com/anthroplankton/quant-finance-project/
